# Power-law Hawkes: current state versus filtered Markov approximation

The true rollout kernel is

$$
\phi(u)=b a^b (u+a)^{-(1+b)}, \qquad u>0,
$$

and the simulator evaluates its full event history directly. The maintained comparison deliberately uses only two observation protocols:

- **Filtered**: current `(t, X_t)` plus the exponential filter bank. This is the proposed finite-dimensional Markov approximation.
- **Current**: only `(t, X_t)`, with no explicit representation of Hawkes memory.

Both protocols are applied to **CT-DDPG, SAC, and DDPG**. History-window variants and PPO are omitted from the reported experiment. The DGM/HJB benchmark knows the fitted exponential mixture and all model parameters. The static comparator is selected independently on validation paths and then frozen before common-path testing.

Every filtered learner receives the same `K=20` exponential readouts; no method is given extra Hawkes state variables. Within each algorithm, current and filtered policies use the same simulator, objective, and test paths. A filtered policy chooses its action from the observed exponential readouts, and that action drives the one observed full-history power-law system. RL is never trained against a counterfactual policy that can see the true history state. Checkpoints are evaluated on a reset common-random validation block. The plotted 90% intervals describe Monte Carlo path uncertainty conditional on one trained policy; a paper-level uncertainty statement should also aggregate independent training seeds.

The final cost plot reports the fully tuned policies. A separate budget plot reports those same common-test costs against the complete training-transition budget, so CT-DDPG's sample efficiency can be assessed without weakening SAC or DDPG.

## Model

The true power-law kernel is

$$
\phi(u)=b a^b (u+a)^{-(1+b)}.
$$

For CT-DDPG features and the DGM/HJB approximation we use

$$
\phi^K(u) = \sum_{k=1}^K w_k e^{-\beta_k u},
\qquad
dZ_t^k=-\beta_k Z_t^k\,dt+dN_t.
$$

The true simulator is the classical additive Hawkes model and has no link or total-intensity cap:

$$
\lambda_t^a = \mu(t,X_{t-},a_t)
+ \alpha q(a_t) \int_0^{t-} \phi(t-s)\,dN_s,
\qquad
q(a)=1-c_a\frac{a}{a_{1/2}+a}.
$$

Only the finite signed approximation applies the positive part to its aggregated memory:

$$
\lambda_t^{K,a} = \mu(t,X_{t-},a_t)
+ \left[\alpha q(a_t) \sum_{k=1}^K w_k Z_{t-}^k\right]_+.
$$

The controlled jump-diffusion is

$$
 dX_t=(b_0-\kappa X_t-b_a a_t)dt
 +[\sigma_0+\sigma_xX_t+\sigma_a a_t]_+dW_t
 +\gamma(t,X_{t-},a_t)dN_t.
$$

The reward has no event-count term:

$$
 r_t=-\left(c_xX_t^2+c_a a_t^2\right),
 \qquad G(X_T)=-c_T X_T^2.
$$

The DGM/PINN solves the HJB over `(t, X, Z^1, ..., Z^K)` and knows the fitted exponential mixture and every coefficient, so it is called the known-model approximation oracle. A filtered RL policy sees only its observed filter state. Its actions drive the single observed full-history system used for training and evaluation; no hidden-memory controller supplies a counterfactual training path.

The baseline coefficient and jump amplitude are bounded, but the total intensity is not capped. Here `alpha=0.99` and the power-law kernel has unit infinite-horizon mass, so the true model is near-critical but subcritical. The short scale `a=0.12` produces visible endogenous clusters, while `b=0.80` keeps the tail non-exponential. The final cell verifies that the signed finite approximation also has finite-horizon envelope mass below one and stress-tests uncapped paths.

In [ ]:
# Tunable parameters for this experiment.
# Cached tuned checkpoints load immediately. Set a FORCE_RETRAIN flag after changing parameters.
from dataclasses import asdict
from pathlib import Path
import sys
import json
import csv
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "classical model with positive link", cwd.parent / "classical model with positive link"]
EXP_DIR = next(
    (
        p if p.name == "powerlaw_kernel_experiment" else p / "powerlaw_kernel_experiment"
        for p in candidates
        if (p / "ct_ddpg_powerlaw.py").exists() or (p / "powerlaw_kernel_experiment" / "ct_ddpg_powerlaw.py").exists()
    ),
    cwd / "powerlaw_kernel_experiment",
)
PROJECT_ROOT = next(
    (p for p in (EXP_DIR, *EXP_DIR.parents) if (p / "neural_numpy.py").exists() and (p / "replay.py").exists()),
    EXP_DIR.parent,
)
FIG_DIR = EXP_DIR / "figures"
RESULT_DIR = EXP_DIR / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)


def write_history_csv(path, rows):
    if not rows:
        return
    fieldnames = sorted({key for row in rows for key in row})
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def observation_label(mode):
    return "filtered" if mode == "filters" else mode


def cached_environment_matches(metadata_paths, expected):
    expected_normalized = json.loads(json.dumps(expected, sort_keys=True))
    for metadata_path in metadata_paths:
        if not metadata_path.exists():
            return False
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        stored = json.loads(json.dumps(metadata.get("environment", {}), sort_keys=True))
        if stored != expected_normalized:
            return False
    return True


for module_path in (PROJECT_ROOT, EXP_DIR, EXP_DIR.parent):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

ENV_PARAMS = dict(
    horizon=8.0,
    dt=0.02,
    beta=0.02,
    kernel_a=0.12,
    kernel_b=0.80,
    filter_betas=tuple(1.00 * k for k in range(1, 21)),
    mixture_weights=None,
    mu0=0.60,
    mu_x=0.002,
    mu_a=-0.002,
    excitation=0.99,
    control_effect_strength=1.08,
    action_half_saturation=0.10,
    baseline_min=1e-6,
    baseline_max=5.0,
    drift_constant=0.0,
    drift_reversion=1.00,
    control_drift_strength=0.02,
    diffusion=0.03,
    volatility_x_strength=0.002,
    control_volatility_strength=0.005,
    jump_size=0.50,
    jump_size_x_strength=0.002,
    jump_size_action_strength=-0.05,
    jump_size_min=1e-4,
    jump_size_max=0.90,
    jump_mark_positive_probability=1.0,
    state_cost=1.00,
    control_cost=0.50,
    terminal_x_cost=0.50,
    x_scale=1.0,
    z_scale=1.0,
)

CT_OBSERVATION_MODES = ("current", "filters")
CT_TRAINING = dict(
    episodes=180,
    warmup_episodes=30,
    updates_per_episode=10,
    eval_every=20,
    eval_episodes=120,
    checkpoint_seed=4019,
    early_stop_patience=6,
    validation_episodes=320,
    validation_seed=8100,
    seed=19,
    blas_threads=2,
    lr_actor=5e-6,
    lr_actor_profile=0.0,
    actor_lr_final_scale=1.0,
    lr_critic=6e-4,
    tau=0.003,
    initial_action=0.18,
    actor_t_skip=0.0,
    actor_x_skip=2.0,
    actor_z_skip=0.0,
    actor_z_slope_skip=0.0,
    actor_z_curve_skip=0.0,
    actor_bias_shift=0.0,
    actor_update_every=1,
)
CT_TRAINING_BY_MODE = {
    "filters": dict(
        episodes=180,
        warmup_episodes=30,
        updates_per_episode=10,
        lr_actor=5e-7,
        lr_actor_profile=0.0,
        actor_lr_final_scale=1.0,
        tau=0.003,
        initial_action=0.18,
        actor_t_skip=0.0,
        actor_x_skip=0.0,
        actor_z_skip=2.0,
        eval_episodes=120,
        actor_z_slope_skip=2.0,
        actor_z_curve_skip=0.0,
        actor_bias_shift=-0.30,
        actor_update_every=2,
        seed=29,
    ),
}

DGM_TRAINING = dict(
    steps=1200,
    batch_size=512,
    hidden_dim=96,
    num_layers=3,
    lr=1.5e-3,
    sampling_mode="reachable",
    reachable_episodes=120,
    uniform_sample_fraction=0.12,
    validation_every=150,
    validation_episodes=64,
    validation_seed=7123,
    early_stop_patience=6,
    restore_best=True,
)

SB3_JOB_CONFIGS = {
    "sac_current": dict(algorithm="sac", observation="current", feature_mode="matched", timesteps=100_000, learning_rate=2e-4, batch_size=512, gradient_steps=1, tau=0.005, ent_coef="auto_0.02", weight_decay=1e-5, eval_freq=12_500, eval_episodes=64, early_stop_patience=3),
    "ddpg_current": dict(algorithm="ddpg", observation="current", feature_mode="matched", timesteps=120_000, learning_rate=5e-5, batch_size=1024, gradient_steps=1, tau=0.003, noise=0.05, weight_decay=1e-4, eval_freq=10_000, eval_episodes=128, early_stop_patience=0, reward_scale=10.0, learning_starts=15_000, initial_action=0.30),
    "sac_filters": dict(algorithm="sac", observation="filters", feature_mode="matched", timesteps=200_000, learning_rate=1e-4, batch_size=512, gradient_steps=2, n_steps=1, tau=0.005, ent_coef="auto_0.005", weight_decay=1e-5, eval_freq=10_000, eval_episodes=96, early_stop_patience=0, warm_start_key="sac_current"),
    "ddpg_filters": dict(algorithm="ddpg", observation="filters", feature_mode="matched", timesteps=160_000, learning_rate=5e-5, batch_size=1024, gradient_steps=1, tau=0.003, noise=0.03, weight_decay=1e-4, eval_freq=10_000, eval_episodes=128, early_stop_patience=0, reward_scale=10.0, learning_starts=15_000, warm_start_key="ddpg_current"),
}

RUN_SB3 = True
FORCE_RETRAIN_CT = False
FORCE_RETRAIN_DGM = False
FORCE_RETRAIN_SB3 = False
CT_PARALLEL_WORKERS = 1
SB3_PARALLEL_WORKERS = 4

VALIDATION_EPISODES = 800
PATH_EPISODES = 20
STATIC_ACTIONS = [0.34]
STATIC_SELECTION_ACTIONS = (0.32, 0.33, 0.34)
STATIC_SELECTION_SEED = 1731
STATIC_SELECTION_EPISODES = 1600
STATIC_PROJECTION_STEP = 0.01
FORCE_RESELECT_STATIC = False
SURFACE_GRID_SIZE = 25
SURFACE_MC_EPISODES = 4
SEED = 999

In [ ]:
from powerlaw_env import PowerLawHawkesConfig, PowerLawHawkesEnv, powerlaw_kernel
from ct_ddpg_powerlaw import collect_episode, evaluate_agent
from dgm_powerlaw_oracle import (
    DGMConfig,
    DGMOraclePolicy,
    evaluate_dgm_policy,
    load_dgm_oracle,
    save_dgm_oracle,
    train_dgm_oracle,
)
from plotting_powerlaw import (
    plot_benchmark_returns,
    plot_ct_history,
    plot_dgm_history,
    plot_episode_ensemble_comparison,
    plot_filtered_training_budget,
    plot_observation_mode_costs,
    plot_policy_mc_cost_surface,
    plot_sb3_training_histories,
    plot_value_surface,
    realized_remaining_cost,
)
from sb3_baselines_powerlaw import collect_sb3_episode, evaluate_sb3_models, sb3_available
from train_tuned_ct_powerlaw_suite import load_tuned_ct_powerlaw_suite, train_tuned_ct_powerlaw_suite
from train_tuned_sb3_powerlaw_suite import (
    load_tuned_powerlaw_suite,
    make_tuned_jobs,
    train_tuned_powerlaw_suite,
)
from stability_diagnostics import (
    plot_stability_certificate,
    sufficient_condition_report,
    uncapped_path_diagnostics,
)
from static_control_selection import select_static_control, plot_static_control_selection

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch: {torch.__version__}; DGM device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except Exception as exc:
    DEVICE = "cpu"
    print(f"PyTorch check failed, using CPU: {exc}")

if RUN_SB3 and not sb3_available():
    print("stable-baselines3 is not installed in this kernel. Install it with: python -m pip install stable-baselines3")

env_cfg = PowerLawHawkesConfig(**ENV_PARAMS)
dgm_cfg = DGMConfig(
    **DGM_TRAINING,
    device=DEVICE,
    print_every=max(1, DGM_TRAINING["steps"] // 10),
)
sb3_jobs = make_tuned_jobs(SB3_JOB_CONFIGS)

filter_obs_dim = 2 + len(env_cfg.filter_betas)
print(f"steps per episode: {env_cfg.steps}; physical state_dim={env_cfg.state_dim}")
print(f"observation dimensions: current=2, filtered={filter_obs_dim}")
print("mixture betas:", env_cfg.filter_betas)
print("mixture weights:", tuple(round(float(w), 6) for w in env_cfg.weights))
print("Execution: CT-DDPG/SB3 use bounded CPU workers; DGM uses CUDA when available.")

In [ ]:
# Visualize the power-law kernel and its exponential-mixture approximation.
u = np.linspace(0.0, env_cfg.horizon, 300)
true_phi = powerlaw_kernel(u, env_cfg.kernel_a, env_cfg.kernel_b)
approx_phi = sum(w * np.exp(-beta * u) for w, beta in zip(env_cfg.weights, env_cfg.filter_betas))
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.plot(u, true_phi, label="power-law kernel")
ax.plot(u, approx_phi, "--", label="signed exponential mixture")
ax.set_xlabel("lag u")
ax.set_ylabel("kernel value")
ax.set_title("Power-law signed exponential-mixture approximation")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

In [ ]:
# Select the constant benchmark on validation paths, independently of final testing.
# This is the fine profile after a coarse full-interval screen; a local quadratic
# smooths Monte Carlo noise before projection to the 0.01 action grid.
static_selection = select_static_control(
    env_cfg,
    evaluate_agent,
    STATIC_SELECTION_ACTIONS,
    seed=STATIC_SELECTION_SEED,
    episodes=STATIC_SELECTION_EPISODES,
    cache_path=RESULT_DIR / "static_control_selection.json",
    environment_signature=asdict(env_cfg),
    projection_step=STATIC_PROJECTION_STEP,
    force=FORCE_RESELECT_STATIC,
)
STATIC_ACTIONS = [float(static_selection["selected_action"])]
print(
    f"Validation-selected static action: a={STATIC_ACTIONS[0]:.2f} "
    f"(continuous quadratic minimizer {static_selection['continuous_minimizer']:.4f})"
)
fig = plot_static_control_selection(
    static_selection,
    FIG_DIR / "static_control_selection.png",
    title="Power-law static-control validation profile",
)
plt.show()

In [ ]:
# Train or load CT-DDPG under current and filtered observations.
ct_metadata_paths = [RESULT_DIR / f"ct_{mode}.json" for mode in CT_OBSERVATION_MODES]
ct_force = FORCE_RETRAIN_CT or not cached_environment_matches(ct_metadata_paths, asdict(env_cfg))
train_tuned_ct_powerlaw_suite(
    RESULT_DIR,
    force=ct_force,
    max_workers=CT_PARALLEL_WORKERS,
    training=CT_TRAINING,
    training_by_mode=CT_TRAINING_BY_MODE,
)
ct_agents, ct_histories, ct_metadata = load_tuned_ct_powerlaw_suite(RESULT_DIR)

for mode, history in ct_histories.items():
    write_history_csv(RESULT_DIR / f"ct_{mode}_history.csv", history)
    fig = plot_ct_history(history, FIG_DIR / f"ct_{mode}_convergence.png")
    plt.show()

# Theory-aligned aliases for value and kernel diagnostics below.
ct_agent = ct_agents["filters"]

In [ ]:
# Train or load the known-parameter DGM/HJB oracle for the exponential-mixture state.
dgm_checkpoint = RESULT_DIR / "dgm_oracle.pt"
dgm_metadata_path = RESULT_DIR / "dgm_oracle.json"
dgm_force = FORCE_RETRAIN_DGM or not cached_environment_matches([dgm_metadata_path], asdict(env_cfg))
if dgm_force or not dgm_checkpoint.exists():
    dgm_oracle, dgm_history = train_dgm_oracle(env_cfg, dgm_cfg)
    save_dgm_oracle(dgm_checkpoint, dgm_oracle, dgm_history)
    dgm_stats = evaluate_dgm_policy(env_cfg, dgm_oracle, seed=SEED, episodes=VALIDATION_EPISODES)
    dgm_metadata = {
        "environment": asdict(env_cfg),
        "config": asdict(dgm_cfg),
        "training_history": dgm_history,
        "evaluation": dgm_stats,
    }
    dgm_metadata_path.write_text(json.dumps(dgm_metadata, indent=2), encoding="utf-8")
else:
    dgm_oracle, dgm_history = load_dgm_oracle(dgm_checkpoint, device=DEVICE)
    dgm_metadata = json.loads(dgm_metadata_path.read_text(encoding="utf-8"))
    print(f"Loaded cached DGM oracle from {dgm_checkpoint}")
# CUDA accelerates training, while sequential one-state rollout decisions are
# faster on CPU because the batches are too small to amortize GPU launch cost.
dgm_oracle = dgm_oracle.to("cpu")
dgm_oracle.eval()
dgm_policy = DGMOraclePolicy(dgm_oracle)
write_history_csv(RESULT_DIR / "dgm_history.csv", dgm_history)
fig = plot_dgm_history(dgm_history, FIG_DIR / "dgm_hjb_convergence.png")
plt.show()

## SAC/DDPG Baseline Note

The wrapper uses the continuous-time one-step discount `exp(-beta dt)`, scales rewards only during training, and restores the best validation checkpoint. Before every checkpoint comparison, the validation environment is reset to the same seed block, so changes in score reflect the policy rather than a different set of Hawkes paths. SAC and DDPG act internally on `[-1,1]`, mapped to the physical control `[0,1]`. Their filtered policies are warm-started from the matching current-state policy, then trained with the enlarged observation using a predeclared budget. Evaluation always reports the original unscaled objective on common test paths.

In [ ]:
# Train or load SAC/DDPG variants. Filtered variants warm-start from the matching current-state policy.
sb3_models_by_key = {}
sb3_metadata = {}
reuse_cached_sb3_test_rows = False
if RUN_SB3 and sb3_available():
    sb3_metadata_paths = [RESULT_DIR / f"{job.key}.json" for job in sb3_jobs]
    sb3_force = FORCE_RETRAIN_SB3 or not cached_environment_matches(sb3_metadata_paths, asdict(env_cfg))
    train_tuned_powerlaw_suite(
        RESULT_DIR,
        force=sb3_force,
        max_workers=SB3_PARALLEL_WORKERS,
        jobs=sb3_jobs,
    )
    try:
        sb3_models_by_key = load_tuned_powerlaw_suite(RESULT_DIR, jobs=sb3_jobs)
    except ModuleNotFoundError as exc:
        comparison_cache = RESULT_DIR / "comparison.json"
        if sb3_force or not comparison_cache.exists():
            raise
        reuse_cached_sb3_test_rows = True
        print(
            "Could not deserialize the unchanged SB3 checkpoints in this NumPy environment; "
            "reusing their compatible seed-999 test rows from comparison.json."
        )
        print(f"Checkpoint compatibility detail: {exc}")
    sb3_metadata = {
        job.key: json.loads((RESULT_DIR / f"{job.key}.json").read_text(encoding="utf-8"))
        for job in sb3_jobs
    }
    fig = plot_sb3_training_histories(sb3_metadata, FIG_DIR / "sb3_validation_convergence.png")
    plt.show()
else:
    print("Skipping SAC/DDPG baselines in this run.")

In [ ]:
# Evaluate all policies on an untouched common Monte Carlo test seed.
comparison_cache = RESULT_DIR / "comparison.json"
cached_comparison_rows = (
    json.loads(comparison_cache.read_text(encoding="utf-8"))
    if comparison_cache.exists()
    else {}
)
rows = {
    f"ct_ddpg_{mode}": evaluate_agent(env_cfg, agent, seed=SEED, episodes=VALIDATION_EPISODES)
    for mode, agent in ct_agents.items()
}
rows["dgm_hjb_mixture_oracle"] = evaluate_dgm_policy(
    env_cfg,
    dgm_oracle,
    seed=SEED,
    episodes=VALIDATION_EPISODES,
)

for key, (model, mode, history_len, feature_mode) in sb3_models_by_key.items():
    rows[key] = evaluate_sb3_models(
        env_cfg,
        {key: model},
        observation_mode=mode,
        history_len=history_len,
        seed=SEED,
        episodes=VALIDATION_EPISODES,
        feature_mode=feature_mode,
    )[key]

if reuse_cached_sb3_test_rows:
    for key in SB3_JOB_CONFIGS:
        if key not in cached_comparison_rows:
            raise KeyError(f"missing cached common-test statistics for {key}")
        rows[key] = cached_comparison_rows[key]

for a in STATIC_ACTIONS:
    rows[f"constant_{a:.2f}"] = evaluate_agent(
        env_cfg,
        None,
        seed=SEED,
        episodes=VALIDATION_EPISODES,
        constant_action=a,
    )

with (RESULT_DIR / "comparison.json").open("w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2)

print("Mean discounted costs on the untouched test paths (lower is better):")
for key, stats in sorted(rows.items(), key=lambda item: -item[1]["return_mean"]):
    print(f"{key:>26s}: {-stats['return_mean']:.5f} +/- {stats.get('return_ci90', 0.0):.5f} (90% CI)")

static_key = f"constant_{STATIC_ACTIONS[0]:.2f}"
static_cost = -rows[static_key]["return_mean"]
print("\nCost reduction relative to static control (positive is better):")
for key, stats in rows.items():
    if key != static_key:
        print(f"{key:>26s}: {static_cost + stats['return_mean']:+.4f}")

print("\nAction diagnostics:")
for key, stats in rows.items():
    if "action_std" in stats:
        print(f"{key:>26s}: mean={stats['action_mean']:.4f}, std={stats['action_std']:.4f}, min={stats['action_min']:.4f}, max={stats['action_max']:.4f}")

fig = plot_benchmark_returns(rows, FIG_DIR / "benchmark_costs.png")
plt.show()

fig = plot_observation_mode_costs(
    rows,
    FIG_DIR / "current_vs_filtered_costs_expanded.png",
)
plt.show()

ct_filtered_training = {**CT_TRAINING, **CT_TRAINING_BY_MODE.get("filters", {})}
filtered_training_transitions = {
    "ct_ddpg_filters": int(ct_filtered_training["episodes"] * env_cfg.steps),
    "sac_filters": int(SB3_JOB_CONFIGS["sac_filters"]["timesteps"]),
    "ddpg_filters": int(SB3_JOB_CONFIGS["ddpg_filters"]["timesteps"]),
}
print("\nFull training-transition budgets for filtered policies:", filtered_training_transitions)
fig = plot_filtered_training_budget(
    rows,
    filtered_training_transitions,
    FIG_DIR / "filtered_cost_vs_training_budget.png",
)
plt.show()

In [ ]:
# Value surfaces use the theory-aligned filtered CT-DDPG only.
fig = plot_policy_mc_cost_surface(
    ct_agent,
    env_cfg,
    save_path=FIG_DIR / "ct_ddpg_policy_mc_cost_surface_z0.png",
    title="Filtered CT-DDPG policy Monte Carlo cost surface, Z=0",
    z_level=0.0,
    grid_size=SURFACE_GRID_SIZE,
    mc_episodes=SURFACE_MC_EPISODES,
    seed=7100,
)
plt.show()

fig = plot_value_surface(
    lambda times, states: -ct_agent.value_path(times, states),
    env_cfg,
    save_path=FIG_DIR / "ct_ddpg_raw_critic_diagnostic_z0.png",
    title="Filtered CT-DDPG critic diagnostic -V(t, X, Z=0)",
    z_level=0.0,
    grid_size=39,
)
plt.show()

fig = plot_value_surface(
    dgm_oracle.value_np,
    env_cfg,
    save_path=FIG_DIR / "dgm_hjb_cost_surface_z0.png",
    title="DGM/HJB mixture-oracle cost surface U(t, X, Z=0)",
    z_level=0.0,
    grid_size=39,
)
plt.show()


In [ ]:
# Path comparison with 90% confidence intervals and common random-number seeds.
static_action = STATIC_ACTIONS[0]
episode_groups = {
    f"CT-DDPG {observation_label(mode)}": [] for mode in CT_OBSERVATION_MODES
}
episode_groups.update({"DGM/HJB oracle": [], f"static a={static_action:.2f}": []})
for key, (_, mode, _, _) in sb3_models_by_key.items():
    algorithm = key.split("_", 1)[0].upper()
    episode_groups[f"{algorithm} {observation_label(mode)}"] = []
value_path_groups = {key: [] for key in episode_groups}

for i in range(PATH_EPISODES):
    ensemble_seed = 9000 + i
    for mode, agent in ct_agents.items():
        label = f"CT-DDPG {observation_label(mode)}"
        ep = collect_episode(env_cfg, agent, np.random.default_rng(ensemble_seed), exploration_std=0.0)
        episode_groups[label].append(ep)
        value_path_groups[label].append(realized_remaining_cost(ep, env_cfg))

    dgm_ep = collect_episode(env_cfg, dgm_policy, np.random.default_rng(ensemble_seed), exploration_std=0.0)
    static_ep = collect_episode(env_cfg, None, np.random.default_rng(ensemble_seed), constant_action=static_action)
    episode_groups["DGM/HJB oracle"].append(dgm_ep)
    episode_groups[f"static a={static_action:.2f}"].append(static_ep)
    value_path_groups["DGM/HJB oracle"].append(realized_remaining_cost(dgm_ep, env_cfg))
    value_path_groups[f"static a={static_action:.2f}"].append(realized_remaining_cost(static_ep, env_cfg))

    for key, (model, mode, history_len, feature_mode) in sb3_models_by_key.items():
        algorithm = key.split("_", 1)[0].upper()
        label = f"{algorithm} {observation_label(mode)}"
        ep = collect_sb3_episode(
            env_cfg,
            model,
            observation_mode=mode,
            history_len=history_len,
            seed=ensemble_seed,
            feature_mode=feature_mode,
        )
        episode_groups[label].append(ep)
        value_path_groups[label].append(realized_remaining_cost(ep, env_cfg))

fig = plot_episode_ensemble_comparison(
    episode_groups,
    value_path_groups,
    FIG_DIR / "policy_path_ensemble_ci.png",
    ci=0.90,
    kernel_weights=env_cfg.weights,
)
plt.show()


In [ ]:
# Optional: show the latest saved PNGs from disk.
# from IPython.display import Image, display
# for name in [
#     "ct_current_convergence.png",
#     "ct_filters_convergence.png",
#     "dgm_hjb_convergence.png",
#     "sb3_validation_convergence.png",
#     "benchmark_costs.png",
#     "current_vs_filtered_costs_expanded.png",
#     "filtered_cost_vs_training_budget.png",
#     "ct_ddpg_policy_mc_cost_surface_z0.png",
#     "ct_ddpg_raw_critic_diagnostic_z0.png",
#     "dgm_hjb_cost_surface_z0.png",
#     "policy_path_ensemble_ci.png",
#     "ct_kernel_approximation.png",
#     "assumption_3_2_uncapped_diagnostic.png",
# ]:
#     path = FIG_DIR / name
#     if path.exists():
#         print(path)
#         display(Image(filename=str(path)))

In [ ]:
# Re-running this notebook loads cached checkpoints. Toggle a FORCE_RETRAIN flag in the parameter cell
# to retrain only CT-DDPG, DGM/HJB, or the SAC/DDPG suite after changing its parameters.

## Kernel Approximation Diagnostic

The rollout simulator uses the true history-based power-law memory, while CT-DDPG and the DGM/HJB oracle use finite exponential filters on the arithmetic beta grid. This diagnostic compares the global signed preselected kernel fit with an ex post signed on-policy memory fit. Both are interpreted through the positive memory link in the approximated intensity; the fitted on-policy weights are diagnostic only and are not used by CT-DDPG training.


In [ ]:
import importlib
import kernel_diagnostics_powerlaw as kdp

# Reload so a notebook kernel does not reuse an older diagnostic module.
kdp = importlib.reload(kdp)

KERNEL_DIAGNOSTIC_EPISODES = 40
fig, kernel_summary = kdp.plot_kernel_approximation_diagnostic(
    env_cfg,
    ct_agents["filters"],
    save_path=FIG_DIR / "ct_kernel_approximation.png",
    episodes=KERNEL_DIAGNOSTIC_EPISODES,
    seed=7300,
)
print("global signed preselected weights:", np.round(kernel_summary["preselected_weights"], 4))
print("on-policy signed memory-fit weights:", np.round(kernel_summary["onpolicy_weights"], 4))
print({k: v for k, v in kernel_summary.items() if "weights" not in k})
plt.show()

# Proposition 3.6 supplies a sufficient-condition certificate. This stress test
# runs the actual full-history simulator without an intensity or event-count cap.
kernel_masses = kdp.stability_envelope_masses(env_cfg)
assumption_report = sufficient_condition_report(
    **kernel_masses,
    baseline_bound=env_cfg.baseline_max,
    jump_amplitude_bound=env_cfg.jump_size_max,
    p_star=4.0,
)
uncapped_summary = uncapped_path_diagnostics(
    lambda: PowerLawHawkesEnv(env_cfg),
    actions=(0.0, 0.3, 1.0),
    episodes=40,
    seed=9800,
)
print("Assumption 3.2 sufficient-condition report:")
print(json.dumps({k: v for k, v in assumption_report.items()}, indent=2))
for action in ("0.00", "0.30", "1.00"):
    print(action, {k: round(v, 4) for k, v in uncapped_summary[action].items()})
fig = plot_stability_certificate(
    assumption_report,
    uncapped_summary,
    save_path=FIG_DIR / "assumption_3_2_uncapped_diagnostic.png",
    title="Power-law experiment: subcritical certificate and uncapped paths",
)
plt.show()